# P00 — Preflight de Codespaces y Bedrock

Este notebook comprueba el entorno sin introducir todavía conceptos del curso. Ejecuta las celdas en orden.

In [ ]:
import sys, platform
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())

In [ ]:
from pathlib import Path
import sqlite3

start = Path.cwd().resolve()
root = next((p for p in [start, *start.parents] if (p / 'pyproject.toml').exists()), None)
if root is None:
    raise RuntimeError('No se ha encontrado la raíz del repositorio')

db = root / 'data' / 'enterprise.db'
with sqlite3.connect(db) as conn:
    rows = conn.execute('SELECT name, tier, region FROM customers ORDER BY name').fetchall()
rows

## AWS IAM Identity Center y Bedrock

La siguiente celda comprueba la identidad temporal obtenida mediante IAM Identity Center. No se usan API keys permanentes.

In [ ]:
import boto3
from enterprise_genai.config import get_settings

settings = get_settings()
session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=settings.aws_region,
)
identity = session.client('sts').get_caller_identity()
print('AWS account:', identity['Account'])
print('Identity:', identity['Arn'])
print('Profile:', settings.aws_profile)
print('Region:', settings.aws_region)
print('Text model:', settings.bedrock_text_model_id)

bedrock = session.client('bedrock', region_name=settings.aws_region)
models = bedrock.list_foundation_models().get('modelSummaries', [])
print('Foundation models visible:', len(models))

## Primera invocación real

Esta celda realiza una llamada real a Amazon Bedrock mediante la API Converse y muestra también consumo de tokens y latencia.

In [ ]:
from enterprise_genai.models import converse_text

response = converse_text(
    'Resume esta incidencia en una frase: 73 usuarios de ACME no pueden acceder a la VPN desde las 08:15.',
    system_prompt='Eres un asistente de soporte empresarial. Responde de forma breve.',
    temperature=0.2,
    max_tokens=120,
)
print(response['text'])
print('Usage:', response['usage'])
print('Metrics:', response['metrics'])